# Simulating data with the Joint Model

In [14]:
import os

import pandas as pd

import leaspy
from leaspy.io.data import Data

leaspy_root = os.path.dirname(leaspy.__file__)

## Import Data



In [15]:
data_path = os.path.join(leaspy_root, "datasets/data/simulated_data_for_joint.csv")

df = pd.read_csv(data_path, dtype={"ID": str}, sep=";")
display(df.head())

,ID,TIME,EVENT_TIME,EVENT_BOOL,Y0,Y1,Y2,Y3
0,116,78.461,85.5,1,0.44444,0.04,0.0,0.0
1,116,78.936,85.5,1,0.60000,0.00,0.0,0.2
2,116,79.482,85.5,1,0.39267,0.04,0.0,0.2
3,116,79.939,85.5,1,0.58511,0.00,0.0,0.0
4,116,80.491,85.5,1,0.57044,0.00,0.0,0.0


To use the Joint Model in Leaspy, your dataset must include the following columns:

1. **ID** : Patient identifier
2. **TIME** : Time of measurement
3. **EVENT_TIME** : Time of the event
4. **EVENT_BOOL** : Event indicator:
   - `1` if the event occurred
   - `0` if censored
   - `2` if a competing event occurred

For one patient, the event time and event bool are the same for each row.



## Joint Model Initialization and Fitting

In [16]:
from leaspy.models import JointModel

data = Data.from_dataframe(df, "joint")
model = JointModel(name="test_model", nb_events=1)

In [17]:
model.fit(data, "mcmc_saem", seed=1312, n_iter=500, progress_bar=False)

/Users/jv.martini/Documents/leaspy/src/leaspy/models/time_reparametrized.py:288: UserWarning: You did not provide `source_dimension` hyperparameter for multivariate model, setting it to ⌊√dimension⌋ = 2.
  warnings.warn(


 ==> Setting seed to 1312

Fit with `AlgorithmName.FIT_MCMC_SAEM` took: 11s


In [18]:
print(model.parameters)

{'betas_mean': tensor([[ 0.0480, -0.1037],
        [ 0.0460, -0.0607],
        [-0.1639, -0.0937]]), 'log_g_mean': tensor([0.1258, 3.0437, 2.7404, 1.2365]), 'log_rho_mean': tensor([1.4383]), 'log_v0_mean': tensor([-3.1961, -3.8937, -3.7175, -2.6191]), 'n_log_nu_mean': tensor([-1.8398]), 'noise_std': tensor(0.0923, dtype=torch.float64), 'tau_mean': tensor([78.9741], dtype=torch.float64), 'tau_std': tensor([6.0641], dtype=torch.float64), 'xi_std': tensor([0.3907], dtype=torch.float64), 'zeta_mean': tensor([[-0.0493],
        [-0.0068]])}


## Simulation

The parameters for simulating patient visits are defined.
These parameters specify the number of patients, the visit spacing, and the timing variability.

In [19]:
visit_params = {
    "patient_number": 5,
    "visit_type": "random",
    "first_visit_mean": 0.0,
    "first_visit_std": 0.4,
    "time_follow_up_mean": 6,  
    "time_follow_up_std": 1.2,
    "distance_visit_mean": 2 / 12,
    "distance_visit_std": 0.75 / 12,
    "min_spacing_between_visits": 1,
}

In [20]:
df_sim = model.simulate(
    algorithm="joint_simulate",
    features=["Y0", "Y1", "Y2", "Y3"],
    visit_parameters=visit_params,
)



Simulate with `joint_simulate` took: 0s


In [21]:
df_sim = df_sim.data.to_dataframe()

display(df_sim)

,ID,TIME,EVENT_TIME,EVENT_BOOL,Y0,Y1,Y2,Y3
0,0,75.0,78.089966,1,0.398945,8.574029e-27,0.000174,0.306660
1,0,76.0,78.089966,1,0.586157,3.649754e-10,0.100133,0.557653
2,0,77.0,78.089966,1,0.490355,1.679223e-01,0.008893,0.716428
3,0,78.0,78.089966,1,0.549328,3.039731e-03,0.045315,0.868493
4,1,78.0,83.881063,0,0.678155,6.227658e-06,0.000156,0.332730
5,1,79.0,83.881063,0,0.786575,7.340836e-05,0.065990,0.147180
6,1,80.0,83.881063,0,0.859611,5.612579e-06,0.008053,0.237598
7,1,81.0,83.881063,0,0.678765,7.922283e-03,0.167131,0.456069
8,1,82.0,83.881063,0,0.729695,2.420786e-01,0.177947,0.523446
9,1,83.0,83.881063,0,0.834127,1.491239e-01,0.445558,0.824290


In [22]:
len(df_sim[df_sim["EVENT_BOOL"] == 0])

12

In [23]:
df_sim

,ID,TIME,EVENT_TIME,EVENT_BOOL,Y0,Y1,Y2,Y3
0,0,75.0,78.089966,1,0.398945,8.574029e-27,0.000174,0.306660
1,0,76.0,78.089966,1,0.586157,3.649754e-10,0.100133,0.557653
2,0,77.0,78.089966,1,0.490355,1.679223e-01,0.008893,0.716428
3,0,78.0,78.089966,1,0.549328,3.039731e-03,0.045315,0.868493
4,1,78.0,83.881063,0,0.678155,6.227658e-06,0.000156,0.332730
5,1,79.0,83.881063,0,0.786575,7.340836e-05,0.065990,0.147180
6,1,80.0,83.881063,0,0.859611,5.612579e-06,0.008053,0.237598
7,1,81.0,83.881063,0,0.678765,7.922283e-03,0.167131,0.456069
8,1,82.0,83.881063,0,0.729695,2.420786e-01,0.177947,0.523446
9,1,83.0,83.881063,0,0.834127,1.491239e-01,0.445558,0.824290
